In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, coalesce

spark = SparkSession.builder.getOrCreate()

# Silver site (sirf current rows)
sites = spark.read.table(f"{SILVER}.site").filter(col("__CURRENT") == True)
ba = spark.read.table(f"{BRONZE}.ba_location_standardized")

joined = sites.join(ba, (sites.city == ba.raw_city) & (sites.country_code == ba.raw_country), "left")

final_df = joined.select(
    col("site_id"), col("site_name"),
    col("city").alias("raw_city"), col("country_code").alias("raw_country"),
    coalesce(col("std_city"), col("city")).alias("std_city"),
    coalesce(col("std_state"), lit("UNKNOWN")).alias("std_state"),
    coalesce(col("std_country"), col("country_code")).alias("std_country"),
    when(col("std_city").isNotNull(), lit(True)).otherwise(lit(False)).alias("is_standardized")
)

final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.site_standardized")
print("✅ Site standardization complete!")